In [0]:
!pip install -U pypdf
!pip install -U langchain-text-splitters
!pip install -U databricks_langchain
!pip install pandas
!pip install faiss-cpu

In [0]:
dbutils.library.restartPython()

In [0]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [0]:
vol_landing_path="/Volumes/rag_on_databricks/landing/vol_landing/pdfs/"
dbutils.fs.ls(vol_landing_path)

In [0]:

pages=[]

for file in dbutils.fs.ls(vol_landing_path):
    reader=PdfReader(file.path.replace("dbfs:", ""))
    for page_num, page in enumerate(reader.pages,start=1):
        text= page.extract_text()
        pages.append(
            {
                "text":text,
                "page_num":page_num
            }
        )
print(len(pages)) #1201
    

In [0]:
%skip
from langchain_community.document_loaders import PyPDFLoader

# Initialize the loader with the path to your PDF file
loader = PyPDFLoader("path/to/your/document.pdf")

# Load the pages (each page becomes a LangChain Document object)
pages = loader.load()

# You can access the content and metadata of a specific page
print(pages[0].page_content)  # The text content of the page
print(pages[0].metadata)      # Metadata (e.g., source file path, page number)


In [0]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1500,chunk_overlap=300, separators=["\n\n", "\n", " ", ", ",""])
splitter.split_text(pages[0]["text"])

In [0]:
chunks=[]
for i, page in enumerate(pages):
    text=page["text"]
    chunks_subset=splitter.split_text(text)
    for j,chunk in enumerate(chunks_subset):
        chunks.append({
            "chunk":chunk,
            "id": f'chunk_id_{page["page_num"]}_{j}'
        })

print(len(chunks))

In [0]:
#chunks[100]

In [0]:
import pandas as pd
import numpy as np

data = pd.DataFrame(chunks) #Chunks to convert that into dataframe and that df is added in delta table..
data.head()

In [0]:
from databricks_langchain import DatabricksEmbeddings

embedding_model = DatabricksEmbeddings(endpoint="databricks-bge-large-en")

In [0]:
np.array(embedding_model.embed_query("What is the meaning of life ?")).shape

In [0]:
# Generate embeddings for all chunks
import time

chunk_list = data["chunk"].tolist()
batch_size = 50  # Reasonable batch size for Databricks
all_embeddings = []

print(f"Processing {len(chunk_list)} chunks (batch_size={batch_size})\n")
start_time = time.time()

for i in range(0, len(chunk_list), batch_size):
    batch = chunk_list[i:i + batch_size]
    batch_embeddings = embedding_model.embed_documents(batch)
    all_embeddings.extend(batch_embeddings)
    print(f"Processed {min(i + batch_size, len(chunk_list))}/{len(chunk_list)} chunks")
    time.sleep(2)  # 2 second delay between batches to avoid rate limits

embeddings = all_embeddings
elapsed = time.time() - start_time
print(f"\nGenerated {len(embeddings)} embeddings in {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"Dimensions: {len(embeddings[0])}")

In [0]:
np.array(embeddings).shape #embedding to convert that into dataframe and that(data) df is added in delta table..


In [0]:
def normalize(vector):
    norms = np.linalg.norm(vector, axis=1, keepdims=True)
    norms[norms==0] = 1e-12
    normalized_embds = vector / norms  # Make them unit vectors
    return normalized_embds

In [0]:
# Normalize embeddings, then save to Delta table
normalized_embeddings = normalize(np.array(embeddings))

spark_df = spark.createDataFrame([
    (
        chunks[i]['id'],
        chunks[i]['chunk'],
        normalized_embeddings[i].tolist()  #Store normalized embeddings
    )
    for i in range(len(chunks))
], ["id", "text", "embedding"])

# Write to Delta Lake
spark_df.write.format("delta").mode("overwrite").saveAsTable("rag_on_databricks.landing.document_embeddings")

print(f"Saved {len(chunks)} chunks to Delta table")

In [0]:
%sql
select * from rag_on_databricks.landing.document_embeddings

In [0]:
#Build Faiss index for fast search
import faiss

# Load normalized embeddings from Delta table
df = spark.table("rag_on_databricks.landing.document_embeddings").toPandas()
chunks_loaded = df[['id', 'text']].rename(columns={'text': 'chunk'}).to_dict('records')
normalized_chunk = np.array(df['embedding'].tolist())

print(f"Loaded {len(normalized_chunk)} normalized embeddings from Delta table")

dimension = 1024
nlist = 1 #50 #Cluster/group size

quantizer = faiss.IndexFlatIP(dimension)  # Inner product = Cosine Similarity for normalized
index = faiss.IndexIVFFlat(quantizer, dimension, nlist, faiss.METRIC_INNER_PRODUCT)

index.train(normalized_chunk) #how to divide these vectors into 50 cluster
index.add(normalized_chunk) #all your embeddings into those clusters

print(f"Faiss index built with {index.ntotal} vectors")

# Option B: Exact search (simpler, good for <100k vectors)
# index = faiss.IndexFlatIP(dimension)
# index.add(normalized_chunk)


In [0]:
normalized_chunk.shape

In [0]:
# Cell 18: Fast retrieval with Faiss
def retrieve(query, k=4):
    # Embed and normalize query
    query_vector = np.array(embedding_model.embed_query(query)).reshape(1, -1)
    normalized_query = normalize(query_vector)
    
    # Search Faiss index
    scores, indices = index.search(normalized_query, k)
    
    # Return results (using chunks_loaded from Delta table)
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "chunk": chunks_loaded[idx]['chunk'],
            "id": chunks_loaded[idx]['id'],
            "score": float(scores[0][i])
        })
    
    return results

In [0]:
from databricks_langchain import ChatDatabricks

model = ChatDatabricks(
    endpoint="databricks-meta-llama-3-1-8b-instruct",
    max_tokens=500,
    temperature=0.1
)

In [0]:
def create_prompt(retrieved_sources, question):
    context = "\n\n".join(
        [
            f"Source {i+1}\n{doc['chunk']}"
            for i, doc in enumerate(retrieved_sources)
        ]
    )

    prompt = f"""
You are an expert AI assistant.

Use ONLY the information provided in the context below to answer the user's question.

Rules:
- Answer only from the provided context.
- Do not make up information.
- If the answer is not available in the context, reply:
  "I couldn't find the answer in the provided documents."
- Keep the answer clear, concise, and professional.
- Use bullet points whenever appropriate.

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

In [0]:

def RAG(query):
    retrieved_sources = retrieve(query, k=4)

    prompt = create_prompt(retrieved_sources, query)

    response = model.invoke(prompt)

    answer=response.content
    return {
        "question": query,
        "answer": answer,
        "sources": retrieved_sources
    }

user_query = "DCP Architecture ?"#"Detailed DCP Architecture: Inbound & Outbound Data Flow ?" 
#"what is the big data in the hadoop ecosystem ?" 
# "What is the YARN Architecture ?"
# "what is the big data in the hadoop ecosystem ?" 
# "what is the capital of france ?"
# "What is sql language related with hadoop and data engineering context ?"
# "What is hdfs in hadoop ?"
# "What is mapreduce in the hadoop ecosystem ?"

result=RAG(user_query)


In [0]:
#check on PII docs

In [0]:
print(result["answer"])